<a href="https://colab.research.google.com/github/elandler/repo-pruebas/blob/main/Ejercicio_M3_U2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# ==========================================
# Instalación de librerías
# ==========================================

!pip install kagglehub[pandas-datasets] textblob spacy -q
!python -m textblob.download_corpora -q

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.


In [22]:
import pandas as pd
import numpy as np

import kagglehub
from kagglehub import KaggleDatasetAdapter

from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [23]:
# ==========================================
# Carga del dataset
# ==========================================

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "vivekhn/yelp-reviews",
    "yelp.csv"
)

df.head()

/tmp/ipykernel_602/2167787502.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'yelp-reviews' dataset.


,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1,2,0
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0


In [24]:
df.info()

df.columns

df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   business_id  10000 non-null  object
 1   date         10000 non-null  object
 2   review_id    10000 non-null  object
 3   stars        10000 non-null  int64 
 4   text         10000 non-null  object
 5   type         10000 non-null  object
 6   user_id      10000 non-null  object
 7   cool         10000 non-null  int64 
 8   useful       10000 non-null  int64 
 9   funny        10000 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 781.4+ KB


(10000, 10)

In [25]:
# Preparación del dataset

df_sent = df[["text", "stars"]].dropna().copy()

# Eliminamos reseñas neutras de 3 estrellas
df_sent = df_sent[df_sent["stars"] != 3]

# Sentimiento real: 1 positivo, 0 negativo
df_sent["sentimiento_real"] = df_sent["stars"].apply(lambda x: 1 if x >= 4 else 0)

df_sent["sentimiento_real"].value_counts()

,count
sentimiento_real,
1,6863
0,1676


In [26]:
# División train/test

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_sent,
    test_size=0.25,
    random_state=42,
    stratify=df_sent["sentimiento_real"]
)

test_df = test_df.copy()

test_df.shape

(2135, 3)

In [27]:
# Método 1: enfoque basado en léxico

positive_words = {
    "excellent", "good", "great", "amazing", "perfect", "best", "love",
    "delicious", "friendly", "nice", "awesome", "wonderful", "fantastic",
    "recommend", "happy", "fresh", "clean", "fast", "favorite"
}

negative_words = {
    "bad", "terrible", "awful", "worst", "horrible", "poor", "slow",
    "dirty", "rude", "disappointed", "hate", "cold", "expensive",
    "bland", "disgusting", "never", "problem", "wrong", "waste"
}

def predecir_lexico(texto):
    texto = str(texto).lower()
    palabras = texto.split()

    pos = sum(1 for p in palabras if p.strip(".,!?;:()[]") in positive_words)
    neg = sum(1 for p in palabras if p.strip(".,!?;:()[]") in negative_words)

    return 1 if pos >= neg else 0

test_df["pred_lexico"] = test_df["text"].apply(predecir_lexico)

test_df[["text", "sentimiento_real", "pred_lexico"]].head()

,text,sentimiento_real,pred_lexico
7651,confirmed better than subway blimpie quiznos e...,1,1
6343,Spring Rolls - Great! Singapore Noodles - Gre...,1,1
7583,I'm so sad I didn't discover the Herb Box soon...,1,1
2683,"Sauce is my kind of place. Great, inexpensive...",1,1
3716,I have been a long time customer at this car w...,0,1


In [28]:
# Método 2: TextBlob

from textblob import TextBlob

def predecir_textblob(texto):
    polarity = TextBlob(str(texto)).sentiment.polarity
    return 1 if polarity >= 0 else 0

test_df["pred_textblob"] = test_df["text"].apply(predecir_textblob)

test_df[["text", "sentimiento_real", "pred_textblob"]].head()

,text,sentimiento_real,pred_textblob
7651,confirmed better than subway blimpie quiznos e...,1,1
6343,Spring Rolls - Great! Singapore Noodles - Gre...,1,1
7583,I'm so sad I didn't discover the Herb Box soon...,1,1
2683,"Sauce is my kind of place. Great, inexpensive...",1,1
3716,I have been a long time customer at this car w...,0,0


In [29]:
# Métricas

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def calcular_metricas(y_true, y_pred, modelo):
    return {
        "modelo": modelo,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

resultados = [
    calcular_metricas(test_df["sentimiento_real"], test_df["pred_lexico"], "Lexico"),
    calcular_metricas(test_df["sentimiento_real"], test_df["pred_textblob"], "TextBlob")
]

df_metricas = pd.DataFrame(resultados)
df_metricas

,modelo,accuracy,precision,recall,f1
0,Lexico,0.839344,0.848300,0.974359,0.906970
1,TextBlob,0.841218,0.850739,0.973193,0.907855


In [30]:
# Reportes

print("Reporte - Método basado en léxico")
print(classification_report(
    test_df["sentimiento_real"],
    test_df["pred_lexico"],
    target_names=["Negativo", "Positivo"],
    zero_division=0
))

print("\nReporte - TextBlob")
print(classification_report(
    test_df["sentimiento_real"],
    test_df["pred_textblob"],
    target_names=["Negativo", "Positivo"],
    zero_division=0
))

Reporte - Método basado en léxico
              precision    recall  f1-score   support

    Negativo       0.73      0.29      0.41       419
    Positivo       0.85      0.97      0.91      1716

    accuracy                           0.84      2135
   macro avg       0.79      0.63      0.66      2135
weighted avg       0.83      0.84      0.81      2135


Reporte - TextBlob
              precision    recall  f1-score   support

    Negativo       0.73      0.30      0.43       419
    Positivo       0.85      0.97      0.91      1716

    accuracy                           0.84      2135
   macro avg       0.79      0.64      0.67      2135
weighted avg       0.83      0.84      0.81      2135



In [31]:
# Comparación final

df_metricas

,modelo,accuracy,precision,recall,f1
0,Lexico,0.839344,0.848300,0.974359,0.906970
1,TextBlob,0.841218,0.850739,0.973193,0.907855
